In [ ]:
from glob import glob

paths = glob('../data/**/*', recursive=True)

In [3]:
import os
only_pdf_files = [f for f in paths if f.endswith('.pdf') and os.path.isfile(f)]

print(len(only_pdf_files))

37237


In [16]:
import os
import csv
import logging
import fitz  # PyMuPDF
from tqdm import tqdm

# Configuração do logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")


def is_born_digital(pdf_path: str, text_threshold: int = 10, ratio_threshold: float = 0.5) -> bool:
    """
    Verifica se um arquivo PDF é nato-digital baseado na extração de texto.

    Parâmetros:
        pdf_path (str): Caminho para o arquivo PDF.
        text_threshold (int): Número mínimo de caracteres para considerar que uma página contém texto significativo.
        ratio_threshold (float): Proporção mínima de páginas com texto para considerar o PDF nato-digital.

    Retorna:
        bool: True se o PDF for nato-digital, False caso contrário.
    """
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        logging.error("Erro ao abrir o arquivo '%s': %s", pdf_path, e)
        return False

    total_pages = doc.page_count
    if total_pages == 0:
        logging.warning("O arquivo '%s' não possui páginas.", pdf_path)
        return False

    pages_with_text = 0
    for page in doc:
        try:
            text = page.get_text("text").strip()
        except Exception as e:
            logging.error("Erro ao extrair texto da página %d em '%s': %s", page.number, pdf_path, e)
            continue
        if len(text) >= text_threshold:
            pages_with_text += 1

    ratio = pages_with_text / total_pages
    return ratio >= ratio_threshold


def generate_pdf_csv(only_pdf_files: list[str], output_csv: str) -> None:
    """
    Processa uma lista de arquivos PDF e gera um CSV com as colunas:
      - URL, mime-type, nato-digital, numero-de-paginas e megabytes.
    
    Parâmetros:
        only_pdf_files (list[str]): Lista de caminhos dos arquivos PDF.
        output_csv (str): Caminho para o arquivo CSV de saída.
    """
    header = ["URL", "mime-type", "nato-digital", "numero-de-paginas", "megabytes"]
    data_rows = []

    # Utiliza tqdm para acompanhar o processamento dos arquivos
    for pdf_file in tqdm(only_pdf_files, desc="Processando PDFs"):
        if not os.path.isfile(pdf_file):
            logging.warning("Arquivo não encontrado: %s", pdf_file)
            continue

        try:
            doc = fitz.open(pdf_file)
            num_pages = doc.page_count
            file_size_mb = os.path.getsize(pdf_file) / (1024 * 1024)
            born_digital = is_born_digital(pdf_file)
        except Exception as e:
            logging.error("Erro ao processar '%s': %s", pdf_file, e)
            continue

        row = [
            pdf_file,                  # URL: nome do arquivo
            "application/pdf",         # mime-type
            born_digital,              # nato-digital: True se nato-digital, False se digitalizado
            num_pages,                 # numero-de-paginas
            round(file_size_mb, 2)     # megabytes, arredondado para 2 casas decimais
        ]
        data_rows.append(row)

    # Escreve os dados coletados no arquivo CSV
    try:
        with open(output_csv, mode="w", newline="", encoding="utf-8") as csv_file:
            writer = csv.writer(csv_file)
            writer.writerow(header)
            writer.writerows(data_rows)
        logging.info("CSV gerado com sucesso: %s", output_csv)
    except Exception as e:
        logging.error("Erro ao escrever o CSV '%s': %s", output_csv, e)

In [19]:
output_csv = "icd.csv"
generate_pdf_csv(only_pdf_files, output_csv)


Processando PDFs:   2%|▏         | 562/37237 [02:05<1:17:27,  7.89it/s]

MuPDF error: format error: No default Layer config



Processando PDFs:   4%|▎         | 1358/37237 [05:30<2:17:14,  4.36it/s]ERROR: Erro ao processar 'data\esaj.tjsp.jus.br\ctoPtl\visualisarContrato.do\nuTitulo_10878\__contains__\esaj.tjsp.jus.br\ctoPtl\downloadDocumento.do\hashCdDocumento_qmQq88N8mr_hashNuTitulo_0YonjPro4G.pdf': Failed to open file 'data\\esaj.tjsp.jus.br\\ctoPtl\\visualisarContrato.do\\nuTitulo_10878\\__contains__\\esaj.tjsp.jus.br\\ctoPtl\\downloadDocumento.do\\hashCdDocumento_qmQq88N8mr_hashNuTitulo_0YonjPro4G.pdf'.
ERROR: Erro ao processar 'data\esaj.tjsp.jus.br\ctoPtl\visualisarContrato.do\nuTitulo_10879\__contains__\esaj.tjsp.jus.br\ctoPtl\downloadDocumento.do\hashCdDocumento_b1GeOL9gmy_hashNuTitulo_yN1D5NXorK.pdf': Failed to open file 'data\\esaj.tjsp.jus.br\\ctoPtl\\visualisarContrato.do\\nuTitulo_10879\\__contains__\\esaj.tjsp.jus.br\\ctoPtl\\downloadDocumento.do\\hashCdDocumento_b1GeOL9gmy_hashNuTitulo_yN1D5NXorK.pdf'.
Processando PDFs:   4%|▍         | 1493/37237 [05:35<04:44, 125.61it/s] ERROR: Erro ao proce

MuPDF error: syntax error: invalid key in dict

MuPDF error: syntax error: invalid key in dict



Processando PDFs:  39%|███▊      | 14360/37237 [19:35<19:25, 19.64it/s]  

...

Processando PDFs:  72%|███████▏  | 26735/37237 [1:00:36<07:49, 22.39it/s]

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check



Processando PDFs:  82%|████████▏ | 30598/37237 [1:03:42<05:18, 20.82it/s]

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check

MuPDF error: library error: zlib error: incorrect header check



Processando PDFs: 100%|██████████| 37237/37237 [1:38:28<00:00,  6.30it/s]
INFO: CSV gerado com sucesso: icd.csv


In [ ]:
import statistics
def analyze_pdf_csv(csv_path: str) -> None:
    """
    Carrega o CSV com as estatísticas dos PDFs e exibe os resultados.

    Parâmetros:
        csv_path (str): Caminho para o arquivo CSV.
    """
    pdf_urls = []
    nato_digitais_list = []
    paginas_list = []
    megabytes_list = []

    # Tenta abrir e ler o CSV
    try:
        with open(csv_path, mode='r', newline='', encoding='utf-8') as csv_file:
            reader = csv.DictReader(csv_file)
            for row in reader:
                pdf_urls.append(row["URL"])
                # Converte para booleano: assume que no CSV está "True" ou "False"
                nato_digitais_list.append(row["nato-digital"].strip().lower() == "true")
                paginas_list.append(int(row["numero-de-paginas"]))
                megabytes_list.append(float(row["megabytes"]))
    except Exception as e:
        logging.error("Erro ao carregar o CSV '%s': %s", csv_path, e)
        return

    total_pdfs = len(pdf_urls)
    nato_digitais = sum(nato_digitais_list)
    digitalizados = total_pdfs - nato_digitais

    # Estatísticas para número de páginas
    try:
        media_paginas = statistics.mean(paginas_list)
        dp_paginas = statistics.stdev(paginas_list) if len(paginas_list) > 1 else 0
    except Exception as e:
        logging.error("Erro ao calcular estatísticas de páginas: %s", e)
        media_paginas = dp_paginas = 0

    # Estatísticas para tamanho dos arquivos em MB
    try:
        media_mb = statistics.mean(megabytes_list)
        dp_mb = statistics.stdev(megabytes_list) if len(megabytes_list) > 1 else 0
    except Exception as e:
        logging.error("Erro ao calcular estatísticas de tamanho dos arquivos: %s", e)
        media_mb = dp_mb = 0

    # Exibe os resultados
    print("Resultados gerais da coleta:")
    print(f"Total de PDFs: {total_pdfs}")
    print("Distribuição Digital:")
    print(f"{nato_digitais} nato-digitais, {digitalizados} digitalizados.")
    print("Estatísticas descritivas:")
    print(f"Número médio de páginas: {media_paginas:.2f} (±{dp_paginas:.2f} páginas).")
    print(f"Tamanho médio dos arquivos: {media_mb:.2f} MB (DP ±{dp_mb:.2f} MB).")

In [ ]:
csv_file_path = "../data/processed/dataset.csv"
analyze_pdf_csv(csv_file_path)

Resultados gerais da coleta:
Total de PDFs: 37181
Distribuição Digital:
25637 nato-digitais, 11544 digitalizados.
Estatísticas descritivas:
Número médio de páginas: 11.40 (±41.96 páginas).
Tamanho médio dos arquivos: 0.93 MB (DP ±2.70 MB).
